# Dyna-Q

Dyna-Q is a model-based Reinforcement Learning algorithm that expands on tabular Q-Learning. It follows the same principles except that it stores information about the experiences. This information is later used to update the action-value function Q

In [ ]:
import numpy as np
import random

# Environment

Let's create a simple environment that represents a grid-world where the agent can move.

The agent has 4 possible actions:
- Move Up
- Move Right
- Move Down
- Move Left

The task is to reach a certain goal from a start position.

The reward is 1 if the agent reaches the goal and 0 otherwise.

Note: we're not using Gymansium's Agent API, thus the environment does not completely follow the usual conventions!

In [ ]:
# Define the grid world environment
class GridWorld:
    def __init__(self, width, height, start, goal):
        self.width = width
        self.height = height
        self.start = start
        self.goal = goal
        self.state = start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        x, y = self.state
        if action == 0:  # Up
            y = min(y + 1, self.height - 1)
        elif action == 1:  # Right
            x = min(x + 1, self.width - 1)
        elif action == 2:  # Down
            y = max(y - 1, 0)
        elif action == 3:  # Left
            x = max(x - 1, 0)

        self.state = (x, y)
        reward = 1 if self.state == self.goal else 0
        done = self.state == self.goal
        return self.state, reward, done

    @staticmethod
    def act_to_string(action):
        if action == 0:
            return "Up"
        elif action == 1:
            return "Right"
        elif action == 2:
            return "Down"
        elif action == 3:
            return "Left"
        else:
            return "Unknown"

The Dyna-Q algorithm follows the next steps:
1. Initialize $Q(s,a)$ and $Model(s,a)$ for all states $s$ and actions $a$.
2. Loop:
  - $s$ = current state.
  - $a$ = $\epsilon$-greedy(s, Q).
  - Take action $a$, get the reward $r$ and new state $s'$.
  - Update Q-table as usual: $Q(s,a) = Q(s,a)+\alpha(r+\gamma\max_aQ(s',a)-Q(s,a))$
  - Store observation in model: $model(s,a) ← r, s'$
  - Perform the planning step: Loop $n$ times:
    - $s$ ← random selection of a previously observed state.
    - $a$ ← random selection of a previously selected action in $s$.
    - $r, s' = model(s,a)$
    - $Q(s,a) = Q(s,a)+\alpha(r+\gamma\max_aQ(s',a)-Q(s,a)$

In [ ]:
# Dyna-Q algorithm
class DynaQ:
    def __init__(self, env, alpha=0.1, gamma=0.95, epsilon=0.1, planning_steps=5):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.planning_steps = planning_steps
        self.q_table = np.zeros((env.width, env.height, 4))
        self.model = {}

    # we use an epsilon-greedy estrategy for exploration
    def choose_action(self, state, explore=True):
        if explore and (random.uniform(0, 1) < self.epsilon):
            return random.randint(0, 3)
        else:
            x, y = state
            return np.argmax(self.q_table[x, y])

    def update_q(self, state, action, reward, next_state, done):
        x, y = state
        nx, ny = next_state

        # First we perform the model-free Q-Learning step. Notice how this
        # code matches the implementation seen for Q-Learning
        best_next_action = np.argmax(self.q_table[nx, ny])

        td_target = reward
        if not done:
            td_target += self.gamma * self.q_table[nx, ny, best_next_action]

        td_error = td_target - self.q_table[x, y, action]
        self.q_table[x, y, action] += self.alpha * td_error

        # Next we update the model
        self.model[(state, action)] = (reward, next_state)

        # Finally we perform the Planning step:
        for _ in range(self.planning_steps):
            # We select random states
            (s_sim, a_sim), (r_sim, ns) = random.choice(list(self.model.items()))
            sx, sy = s_sim
            nsx, nsy = ns
            # select the best next action and update the Q-table
            best_next_action = np.argmax(self.q_table[nsx, nsy])
            td_target = r_sim if done else r_sim + self.gamma * self.q_table[nsx, nsy, best_next_action]
            td_error = td_target - self.q_table[sx, sy, a_sim]
            self.q_table[sx, sy, a_sim] += self.alpha * td_error

    def train(self, episodes=100):
        for _ in range(episodes):
            state = self.env.reset()
            done = False
            while not done:
                action = self.choose_action(state)
                next_state, reward, done = self.env.step(action)
                self.update_q(state, action, reward, next_state, done)
                state = next_state

In [ ]:
# Set up the environment and agent
env = GridWorld(width=5, height=5, start=(0, 0), goal=(2, 2))
agent = DynaQ(env)

In [ ]:
# Train the agent
agent.train(episodes=100)

In [ ]:
# Print the learned Q-values
print("Learned Q-values:")
print(agent.q_table)

In [ ]:
# Print the learned model
print("Learned model:")
for k, v in agent.model.items():
    print(f"{k} -> {v}")

In [ ]:
state = env.reset()
print(f"Initial state: {state}")
print(f"Final state: {env.goal}")
done = False
while not done:
    action = agent.choose_action(state, False) # Use the learned policy, don't explore
    new_state, reward, done = env.step(action)
    print(f"{state} -> a:{GridWorld.act_to_string(action):>5}, r:{reward} -> {new_state}")
    state = new_state
